# Reward encoding in visual cortex: Zhong et al. 2025

**Question:** as mice learn a visual discrimination task, does the **proportion**
of reward-encoding neurons in each visual-cortical region change, and does their
**activation strength** change — tracked day by day, and compared against an
**unsupervised** cohort that sees identical stimuli but is never rewarded?

**Method:** for each sampled neuron, split its rewarded-corridor trials by
whether the sound cue (whose position tracks the reward zone) fell **early** or
**late**, and compute `d' = 2*(mean_late - mean_early) / (std_late + std_early)`
in the reward zone — Zhong et al.'s own reward-prediction index (their Fig. 4f–g).
The flag is **cross-validated**: a neuron counts only if `|d'| >= 0.3` reproduces
across two interleaved trial folds with the same sign, so read noise doesn't pass.
We use this cue-based index instead of an encoding-model reward *ablation* because
the cue plays whether or not water is delivered — so, unlike an ablation of the
water-delivery regressor (which we tried first; see `docs/method.md` for why it
was retired), the **same index is defined for unsupervised mice too**, making the
supervised-vs-unsupervised contrast a fair one rather than zero-by-construction.

Per (cohort, stage, region, mouse) we get two numbers: the **percentage** of
sampled neurons flagged (recruitment) and their **mean cross-validated |d'|**
(gain). Both are tracked across four stages — `train1_before/after`,
`train2_before/after` — and each **animal** imaged in every stage of its cohort
contributes one connected trajectory, so the final figure shows individual
animals, not just group means.

**Run in Colab:** **Runtime → Run all**. No Drive mount needed; files stream from
the official Figshare deposit and are cached (one SVD + retinotopy file per
mouse, ~100–180 MB each). With the animal counts found in the deposit — 4
supervised, 6 unsupervised, each imaged across all four stages — a full run
downloads ~2 GB and completes in well under an hour on a standard Colab runtime.


## 1. Setup and configuration

Colab already includes the scientific Python packages used below. The configuration
cell is the only place to change sample size, threshold, or random seed.


In [ ]:
from __future__ import annotations

import platform
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import seaborn as sns
try:
    from tqdm.auto import tqdm  # progress bars with ETA (preinstalled in Colab)
except ModuleNotFoundError:  # ponytail: minimal test env lacks tqdm; no-op fallback
    def tqdm(iterable=None, **_):
        return iterable

sns.set_theme(style="whitegrid", context="notebook")
np.set_printoptions(precision=4, suppress=True)


In [ ]:
SEED = 42
DATA_DIR = Path("/content/Zhong_et_al_2025")
OUTPUT_DIR = Path("/content")
ARTICLE_ID = 28811129        # the whole Figshare deposit

AREA_CODES = {
    "V1": (8,),
    "mHV": (0, 1, 2, 9),
    "lHV": (5, 6),
    "aHV": (3, 4),
}

N_PER_REGION = 750           # neurons sampled per region per mouse (capped at available)
DPRIME_THRESHOLD = 0.3       # Zhong et al.'s reward-prediction cutoff, |d'| >= 0.3
N_SHUFFLES = 100              # late/early label shuffles -> empirical chance floor
REWARD_ZONE_DM = (5, 40)     # corridor dm averaged for reward-zone activity (0.5-4 m)

# The learning-time axis: two training blocks, before vs after learning in each.
STAGE_ORDER = ["train1_before", "train1_after", "train2_before", "train2_after"]

# (cohort, stage) -> behavior filename. Both cohorts get the SAME cue-based d'
# index: the sound cue plays whether or not water is delivered. Confirmed on an
# unsupervised file (TX88_2022_06_20_1): RewardFr all-NaN, isRew all-False, but
# SoundPos present with the same range as supervised sessions - a genuine,
# non-degenerate control, unlike the retired reward-ablation approach (which
# ablated water delivery and so scored unsupervised mice zero by construction).
COHORT_STAGE_FILES = {
    ("supervised", "train1_before"): "Beh_sup_train1_before_learning.npy",
    ("supervised", "train1_after"): "Beh_sup_train1_after_learning.npy",
    ("supervised", "train2_before"): "Beh_sup_train2_before_learning.npy",
    ("supervised", "train2_after"): "Beh_sup_train2_after_learning.npy",
    ("unsupervised", "train1_before"): "Beh_unsup_train1_before_learning.npy",
    ("unsupervised", "train1_after"): "Beh_unsup_train1_after_learning.npy",
    ("unsupervised", "train2_before"): "Beh_unsup_train2_before_learning.npy",
    ("unsupervised", "train2_after"): "Beh_unsup_train2_after_learning.npy",
}

print(f"Python {platform.python_version()} | NumPy {np.__version__}")
print(f"Seed={SEED}; {N_PER_REGION} neurons/region; |d'| >= {DPRIME_THRESHOLD}; "
      f"{len(COHORT_STAGE_FILES)} cohort x stage sessions")


## 2. Data access

Files are streamed from the official Figshare deposit and resolved by name against
a one-time manifest fetch, then cached and size-verified: a truncated download
fails loudly instead of producing quiet nonsense. Zhong's SVD files contain an
unused serialized scikit-learn object; this notebook reads only `U` and `V`.


In [ ]:
# --- Data access: resolve any deposit file by name, cached and size-verified ---
def fetch_manifest(article_id=ARTICLE_ID):
    """Map every filename in the Figshare deposit to its id and size (one request)."""
    r = requests.get(f"https://api.figshare.com/v2/articles/{article_id}", timeout=(15, 60))
    r.raise_for_status()
    return {f["name"]: {"id": f["id"], "size": f["size"]} for f in r.json()["files"]}


def download_by_name(name, manifest, data_dir=DATA_DIR):
    """Download one deposit file by exact name; cached and size-verified."""
    data_dir = Path(data_dir); data_dir.mkdir(parents=True, exist_ok=True)
    if name not in manifest:
        raise KeyError(f"{name} is not in the Figshare deposit")
    target = data_dir / name
    expected = manifest[name]["size"]
    if target.exists() and target.stat().st_size == expected:
        return target
    url = f"https://ndownloader.figshare.com/files/{manifest[name]['id']}"
    with requests.get(url, stream=True, timeout=(15, 600)) as r:
        r.raise_for_status()
        with target.open("wb") as fh:
            for chunk in r.iter_content(1024 * 1024):
                fh.write(chunk)
    if target.stat().st_size != expected:
        raise RuntimeError(f"Truncated download for {name}; rerun the cell to retry.")
    return target


## 3. Region mapping and balanced neuron sampling

Region membership comes from the retinotopy file. We sample up to `N_PER_REGION`
neurons from each mapped region with a fixed seed, independently of activity or
model fit, then reconstruct only those neurons from Zhong's 400-component SVD.


In [ ]:
def map_regions(iarea):
    """Map Zhong's numeric area codes onto the four regions we compare."""
    iarea = np.asarray(iarea)
    labels = np.full(len(iarea), "unmapped", dtype=object)
    for name, codes in AREA_CODES.items():
        labels[np.isin(iarea, codes)] = name
    return labels.astype(str)


def balanced_sample(regions, n_per_region, seed=42):
    """Pick up to n_per_region neurons per region, blind to activity and model fit.

    If a region has fewer neurons than requested we take all of them (and say so)
    instead of crashing, so the same call works on tiny and huge sessions alike.
    Regions can therefore end up with different counts - that is fine, because we
    always report *percentages* with a per-region denominator downstream.
    """
    regions = np.asarray(regions)
    rng = np.random.default_rng(seed)
    selected, labels = [], []
    for region in AREA_CODES:
        candidates = np.flatnonzero(regions == region)
        n = min(n_per_region, len(candidates))
        if n == 0:
            print(f"  {region}: no mapped neurons, skipping")
            continue
        if n < n_per_region:
            print(f"  {region}: only {n} neurons available (< {n_per_region} requested)")
        selected.append(np.sort(rng.choice(candidates, n, replace=False)))
        labels.extend([region] * n)
    return np.concatenate(selected), np.asarray(labels)


def reconstruct_selected(svd, selected):
    """Rebuild activity for just the sampled neurons from the 400-component SVD."""
    U = np.asarray(svd["U"], dtype=np.float32)  # components x neurons
    V = np.asarray(svd["V"], dtype=np.float32)  # components x frames
    return (U[:, np.asarray(selected, dtype=int)].T @ V).T.astype(np.float32, copy=False)


## 4. Cross-validated d′(late vs early cue)

Split each rewarded-corridor trial's reward-zone activity by whether the sound
cue fell early or late, and ask how different those two trial groups look
(Zhong et al.'s Fig. 4f–g index). A single `|d'| >= 0.3` on all trials is
deceptively noisy: with only a few dozen trials, pure noise clears 0.3 often. So
we **cross-validate** — split into two interleaved folds (first sorted by the
late/early label so each fold gets a balanced mix), score each independently,
and flag a neuron only if both folds agree in magnitude and sign. Its
**strength** (the activation tracked across days) is the cross-validated `|d'|`
magnitude itself, reported for *all* sampled neurons so it is not biased by
the selection.


In [ ]:
def _dprime_between(trial_activity, group):
    """Signal-detection d' between two groups of trials, per neuron.

    trial_activity: neurons x trials. group: bool over trials (True = group A).
    d' = 2*(mean_A - mean_B) / (std_A + std_B) - the same form used in the paper
    and the 50k loading notebook. The tiny +1e-9 just avoids a divide-by-zero for
    a silent neuron.
    """
    a = trial_activity[:, group]
    b = trial_activity[:, ~group]
    return 2 * (a.mean(1) - b.mean(1)) / (a.std(1) + b.std(1) + 1e-9)


def _reward_encoding(trial_mean, late, threshold):
    """Cross-validated reward encoding per neuron. Returns (flag, strength).

    - flag (for the proportion): True only if |d'(late vs early)| >= threshold in
      both folds with the same sign - a real anticipatory effect reproduces across
      folds; noise almost never does.
    - strength (for the activation): the neuron's cross-validated reward-d' magnitude,
      (|d'_foldA| + |d'_foldB|)/2 - reported for all sampled neurons (not just the
      flagged ones, so it is not biased by the selection).
    """
    order = np.argsort(late)                      # early trials first, then late
    tm, lt = trial_mean[:, order], late[order]
    fold = np.arange(tm.shape[1]) % 2             # interleave -> both folds balanced
    a, b = fold == 0, fold == 1
    d_a = _dprime_between(tm[:, a], lt[a])
    d_b = _dprime_between(tm[:, b], lt[b])
    flag = (np.abs(d_a) >= threshold) & (np.abs(d_b) >= threshold) & (np.sign(d_a) == np.sign(d_b))
    strength = (np.abs(d_a) + np.abs(d_b)) / 2
    return flag, strength


def reward_dprime_session(beh, mouse, manifest,
                          n_per_region=N_PER_REGION,
                          reward_zone=REWARD_ZONE_DM, seed=SEED):
    """Reward encoding per sampled neuron for one mouse, grouped by region.

    Split the rewarded-corridor trials by where the sound cue fell (early vs late),
    average each neuron's activity in the reward zone per trial, and ask how
    different late-cue vs early-cue trials look. `_reward_encoding` turns that into
    a cross-validated flag and a strength (reward-d' magnitude = its "activation").
    Works identically for supervised and unsupervised mice: the cue and reward
    zone exist in both, water delivery does not enter this computation at all.
    """
    stim_id, uniqW, wall_name = beh["stim_id"], beh["UniqWalls"], beh["WallName"]
    if not np.any(stim_id == 2):
        raise ValueError("no rewarded stimulus (leaf1) in this session")
    rewarded_wall = uniqW[stim_id == 2][0]                    # leaf1 = rewarded stimulus
    trial_is_rewarded = wall_name == rewarded_wall
    cue_pos = beh["SoundPos"] if "SoundPos" in beh else np.mod(beh["SoundDelPos"], 60)
    cue_pos = np.asarray(cue_pos, dtype=float)
    trial_frame = np.asarray(beh["ft_trInd"], dtype=float)    # trial index of each frame
    pos_frame = np.asarray(beh["ft_Pos"], dtype=float)        # position in corridor (dm)
    wall_frame = np.asarray(beh["ft_WallID"])                 # stimulus shown each frame
    moving = np.asarray(beh["ft_move"]) > 0

    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", message="Trying to unpickle estimator")
        svd = np.load(download_by_name(f"{mouse}_SVD_dec.npy", manifest),
                      allow_pickle=True).item()
    retin = np.load(download_by_name(f"{mouse[:-1]}trans.npz", manifest), allow_pickle=False)
    sel, sel_regions = balanced_sample(map_regions(retin["iarea"]), n_per_region, seed=seed)
    activity = reconstruct_selected(svd, sel)                 # frames x neurons
    del svd

    # per-trial reward-zone mean activity: neurons x rewarded-trials
    n_frames = min(activity.shape[0], len(trial_frame))
    activity = activity[:n_frames]
    in_zone = ((wall_frame[:n_frames] == rewarded_wall)
               & (pos_frame[:n_frames] >= reward_zone[0])
               & (pos_frame[:n_frames] < reward_zone[1])
               & moving[:n_frames])
    rewarded_trials = np.flatnonzero(trial_is_rewarded)
    frame_trial = np.rint(trial_frame[:n_frames])
    trial_mean = np.full((activity.shape[1], len(rewarded_trials)), np.nan, np.float32)
    for j, t in enumerate(rewarded_trials):
        fr = in_zone & (frame_trial == t)
        if fr.any():
            trial_mean[:, j] = activity[fr].mean(0)

    keep = ~np.isnan(trial_mean).any(0)                       # trials with reward-zone frames
    trial_mean = trial_mean[:, keep]
    cue = cue_pos[rewarded_trials][keep]
    if np.ptp(cue) == 0:
        raise ValueError("cue position never varies - cannot split early vs late")
    late = cue > np.median(cue)                               # late-cue vs early-cue trials
    if late.sum() < 8 or (~late).sum() < 8:                   # need enough for a 2-fold split
        raise ValueError(f"too few early/late-cue trials for cross-validated d' "
                         f"({(~late).sum()} early / {late.sum()} late)")

    flag, strength = _reward_encoding(trial_mean, late, DPRIME_THRESHOLD)
    return {
        "flag_by_region": {r: flag[sel_regions == r] for r in AREA_CODES},
        "strength_by_region": {r: strength[sel_regions == r] for r in AREA_CODES},
        "trial_mean": trial_mean, "late": late,
    }


def run_dprime_synthetic_check():
    """One ramping neuron + noise neurons; the flag must catch the ramp, not the noise."""
    rng = np.random.default_rng(3)
    late = np.arange(60) % 2 == 0
    tm = rng.normal(0, 1.0, (6, 60)).astype(np.float32)
    tm[0, late] += 3.0                       # neuron 0 ramps on late-cue trials
    flag, strength = _reward_encoding(tm, late, DPRIME_THRESHOLD)
    if not (flag[0] and not flag[1:].any()):
        raise AssertionError(f"d' synthetic check failed: flag={flag}")
    return {"flagged": int(flag.sum()), "ramp_strength": round(float(strength[0]), 2)}


## 5. Run across cohorts, stages, and animals

An **animal**, not a recording session, is the replicate: the same mouse is
imaged on a different date for "before" and "after" within a training block, so
we key by the short animal id (`mouse.split("_")[0]`) and only that id needs to
appear in every stage of a cohort for its trajectory to be unbroken. Any animal
missing a stage (too few reward events, no usable early/late split, or simply
absent from that stage's file) just leaves a gap in its own line — it is not
dropped from the others.


In [ ]:
# --- Run cross-validated d' across every (cohort, stage), animal-paired ---
# One SVD + retinotopy file per animal per stage (~100-180 MB, cached after the
# first run). Downloads happen once; rerunning this cell reuses the cache.
print("d' sanity check:", run_dprime_synthetic_check())
manifest = fetch_manifest()


def animal_id(session_key):
    return session_key.split("_")[0]


stage_beh = {
    key: np.load(download_by_name(fname, manifest), allow_pickle=True).item()
    for key, fname in COHORT_STAGE_FILES.items()
}

# Only animals imaged in EVERY stage of a cohort (and with SVD + retinotopy
# files available) give an unbroken before/after trajectory across both
# training blocks.
cohorts = sorted({cohort for cohort, _ in COHORT_STAGE_FILES})
paired_animals = {}
for cohort in cohorts:
    per_stage_ids = [
        {animal_id(m) for m in stage_beh[(cohort, stage)]
         if f"{m}_SVD_dec.npy" in manifest and f"{m[:-1]}trans.npz" in manifest}
        for stage in STAGE_ORDER
    ]
    paired_animals[cohort] = sorted(set.intersection(*per_stage_ids))
    print(f"{cohort}: {len(paired_animals[cohort])} animal(s) paired across all "
          f"4 stages -> {paired_animals[cohort]}")

rows, floor_rows, skipped = [], [], []
rng = np.random.default_rng(SEED)
for cohort in cohorts:
    for stage in STAGE_ORDER:
        beh_all = stage_beh[(cohort, stage)]
        animals = paired_animals[cohort]
        for animal in tqdm(animals, desc=f"{cohort}/{stage}", unit="mouse"):
            mouse = next(m for m in beh_all if animal_id(m) == animal)
            try:
                res = reward_dprime_session(beh_all[mouse], mouse, manifest)
            except (KeyError, ValueError, RuntimeError) as err:
                skipped.append((cohort, stage, animal, str(err)))
                continue
            for region in AREA_CODES:
                flag = res["flag_by_region"][region]
                strength = res["strength_by_region"][region]
                if len(flag):
                    rows.append({
                        "cohort": cohort, "stage": stage, "region": region,
                        "animal": animal, "mouse": mouse,
                        "pct_reward": 100 * np.mean(flag),          # how MANY (recruitment)
                        "mean_dprime": float(np.mean(strength)),    # how STRONGLY (gain)
                        "n_neurons": len(flag),
                    })
            # empirical chance floor: same cross-validated flag on shuffled late/early labels
            tm, late = res["trial_mean"], res["late"]
            shuffled_pct = [
                100 * np.mean(_reward_encoding(tm, rng.permutation(late), DPRIME_THRESHOLD)[0])
                for _ in range(N_SHUFFLES)
            ]
            floor_rows.append({"cohort": cohort, "pct": float(np.mean(shuffled_pct))})

results_df = pd.DataFrame(rows)
chance_floor = (pd.DataFrame(floor_rows).groupby("cohort")["pct"].mean()
               if floor_rows else pd.Series(dtype=float))
n_animals = results_df[["cohort", "animal"]].drop_duplicates().shape[0] if len(results_df) else 0
print(f"\n{len(results_df)} region x animal x stage rows from {n_animals} animal-cohort pairs")
print(f"chance floor (label-shuffle) by cohort:\n{chance_floor}")
if skipped:
    print(f"{len(skipped)} (cohort, stage, animal) skipped - inspect `skipped` for reasons.")
if len(results_df):
    display(results_df.groupby(["cohort", "stage", "region"])[["pct_reward", "mean_dprime"]]
            .mean().round(3))


## 6. Results — proportion and activation across learning, by region and cohort

Two figures: **recruitment** (percentage of neurons flagged as reward-encoding)
and **gain** (their mean cross-validated `|d'|`). Each has one panel per region,
with y-axis bounds shared across all four regions in that figure so heights are
directly comparable. In each panel, thin lines are individual animals (a gap
means that animal's stage was skipped), the box shows the distribution across
animals at that stage, and the bold dot + line is the cohort mean. Supervised
and unsupervised are overlaid so the paper's key dissociation — the
reward-prediction signal should be supervised-only — is visible directly in the
recruitment/gain trajectories, not just inferred from a separate contrast.


In [ ]:
# --- Figures: recruitment and gain, each its own figure, across learning by region ---
COHORT_COLORS = {"supervised": "#4C72B0", "unsupervised": "#DD8452"}
COHORT_OFFSET = {"supervised": -0.12, "unsupervised": 0.12}

# One consistent color per animal (same animal = same color in every panel),
# drawn from viridis across ALL animals of both cohorts so every mouse gets a
# distinct, unique shade; cohort identity still comes through via the box
# color and bold mean line (COHORT_COLORS).
ANIMAL_COLORS = {}
all_animals = sorted((cohort, animal) for cohort in cohorts for animal in paired_animals[cohort])
n_animals = len(all_animals)
for i, (cohort, animal) in enumerate(all_animals):
    ANIMAL_COLORS[(cohort, animal)] = plt.cm.viridis(i / max(n_animals - 1, 1))


def _panel(ax, metric, region, show_floor):
    xpos = {s: i for i, s in enumerate(STAGE_ORDER)}
    for cohort in cohorts:
        color = COHORT_COLORS.get(cohort, "0.3")
        offset = COHORT_OFFSET.get(cohort, 0.0)
        sub = results_df[(results_df.region == region) & (results_df.cohort == cohort)]
        if not len(sub):
            continue
        for animal, grp in sub.groupby("animal"):          # per-animal trajectory (gaps = nan)
            animal_color = ANIMAL_COLORS.get((cohort, animal), color)
            by_stage = grp.set_index("stage")[metric]
            ys = [by_stage.get(s, np.nan) for s in STAGE_ORDER]
            xs = [xpos[s] + offset for s in STAGE_ORDER]
            ax.plot(xs, ys, color=animal_color, lw=1.1, alpha=0.8, zorder=1)
            valid = [(x, y) for x, y in zip(xs, ys) if np.isfinite(y)]
            if valid:                                       # label each mouse at its last point
                x_last, y_last = valid[-1]
                ax.annotate(animal, (x_last, y_last), xytext=(4, 0),
                           textcoords="offset points", color=animal_color,
                           fontsize=8, va="center", zorder=5)
        box_data, box_pos = [], []                          # distribution across animals
        for s in STAGE_ORDER:
            vals = sub.loc[sub.stage == s, metric].to_numpy()
            if len(vals):
                box_data.append(vals); box_pos.append(xpos[s] + offset)
        if box_data:
            bp = ax.boxplot(box_data, positions=box_pos, widths=0.18, patch_artist=True,
                            showfliers=False, zorder=2)
            for patch in bp["boxes"]:
                patch.set_facecolor(color); patch.set_alpha(0.25); patch.set_edgecolor(color)
            for element in ("whiskers", "caps", "medians"):
                for line in bp[element]:
                    line.set_color(color)
        means, xs_mean = [], []                             # bold cohort mean +/- s.e.m.
        for s in STAGE_ORDER:
            vals = sub.loc[sub.stage == s, metric].to_numpy()
            if not len(vals):
                continue
            x = xpos[s] + offset
            sem = vals.std(ddof=1) / np.sqrt(len(vals)) if len(vals) > 1 else 0.0
            ax.errorbar(x, vals.mean(), yerr=sem, fmt="o", color=color, markersize=6,
                       lw=2.5, capsize=4, zorder=4)
            means.append(vals.mean()); xs_mean.append(x)
        if len(xs_mean) > 1:
            ax.plot(xs_mean, means, color=color, lw=2.5, zorder=3)
        if show_floor and cohort in chance_floor.index:
            ax.axhline(chance_floor[cohort], ls="--", color=color, lw=1.2, alpha=0.6)
    ax.set_xticks(range(len(STAGE_ORDER)))
    ax.set_xticklabels([s.replace("_", "\n") for s in STAGE_ORDER], fontsize=11)
    ax.set_xlim(-0.5, len(STAGE_ORDER) - 0.5)
    ax.tick_params(axis="y", labelsize=11)
    ax.spines[["top", "right"]].set_visible(False)


def _make_figure(metric, ylab, show_floor, suptitle):
    """One metric, one figure: 4 region panels with a shared, normalized y-axis."""
    fig, axes = plt.subplots(1, 4, figsize=(20, 5.5))
    for ax, region in zip(axes, AREA_CODES):
        _panel(ax, metric, region, show_floor)
        ax.set_title(region, fontsize=15, fontweight="bold")
    axes[0].set_ylabel(ylab, fontsize=13)

    # normalize: every region panel in this figure shares the same y bounds,
    # so heights are directly comparable across regions.
    lo = min(ax.get_ylim()[0] for ax in axes)
    hi = max(ax.get_ylim()[1] for ax in axes)
    for ax in axes:
        ax.set_ylim(lo, hi)

    handles = [plt.Line2D([], [], color=COHORT_COLORS["supervised"], marker="o", label="supervised"),
              plt.Line2D([], [], color=COHORT_COLORS["unsupervised"], marker="o", label="unsupervised")]
    if show_floor:
        handles.append(plt.Line2D([], [], color="0.4", ls="--", label="chance floor (label shuffle)"))
    axes[-1].legend(handles=handles, fontsize=11, loc="upper left", frameon=False)
    fig.suptitle(suptitle, fontsize=16, y=1.06)

    animal_handles = [
        plt.Line2D([], [], color=c, lw=2.5, label=f"{cohort[:3]}: {animal}")
        for (cohort, animal), c in sorted(ANIMAL_COLORS.items())
    ]
    fig.legend(handles=animal_handles, loc="lower center", bbox_to_anchor=(0.5, -0.1),
              ncol=min(len(animal_handles), 6), fontsize=10, frameon=False, title="animals")

    fig.subplots_adjust(wspace=0.32)
    plt.tight_layout()
    plt.show()


if len(results_df) == 0:
    print("No sessions produced results - check `skipped`.")
else:
    _make_figure("pct_reward", "% reward-encoding neurons\n(cross-val d', |d'|>=0.3)", True,
                "Recruitment: reward-encoding neurons across learning, by region\n"
                "thin colored lines = individual animals (labeled by id), box = "
                "distribution across animals, bold = cohort mean")
    _make_figure("mean_dprime", "mean |d'| (late vs early cue)\n(activation strength)", False,
                "Gain: activation strength of reward-encoding neurons across learning, by region\n"
                "thin colored lines = individual animals (labeled by id), box = "
                "distribution across animals, bold = cohort mean")

    csv_path = OUTPUT_DIR / "reward_encoding_by_animal.csv"
    results_df.to_csv(csv_path, index=False)
    print(f"Saved {csv_path} ({len(results_df)} rows)")
    try:
        from google.colab import files as colab_files
        colab_files.download(str(csv_path))
    except ImportError:
        pass


## Interpretation checklist

- **Recruitment vs gain.** If the proportion of flagged neurons rises with
  learning while their mean `|d'|` stays flat, that's *recruitment* (new
  neurons join the code). If the proportion is flat but `|d'|` rises, that's
  *gain* (the same-sized population responds more strongly). Both can be true
  at once, and region/cohort can differ.
- **Supervised vs unsupervised is the paper's headline test.** Zhong et al.
  report the reward-prediction signal in anterior HVAs only with reward
  (task p=0.0069 vs unsupervised p=0.708, their Fig. 4g). Support for that
  account here looks like: the supervised line sits clearly above its chance
  floor and rises with learning in `aHV`, while the unsupervised line tracks
  its own chance floor throughout.
- **No individual-neuron tracking.** "Before" and "after" are different
  imaging sessions with independent neuron indexing (confirmed against
  `Imaging_Exp_info.npy` — no cell-registration file in the deposit), so a
  rising proportion or `|d'|` reflects the *population*, not the same cells
  strengthening. An animal's trajectory is a paired population-level
  comparison, not a tracked-neuron one.
- **Chance floor, not p<0.05 alone.** The floor is an empirical shuffle of the
  late/early label through the *same* cross-validated flag, per cohort — read
  a line against its own floor, not zero.
- **No multiple-comparison correction** across regions or stages; treat a
  single borderline crossing as "worth a follow-up," not proof.


## 7. Encoding-model cross-check: does a GLM reward-coefficient agree with d′?

**Pilot, single session.** Runs on one post-learning supervised session
(`train2_after`, the first paired animal) to check the idea before scaling to
every session — that's one word away (loop over `paired_animals` and
`STAGE_ORDER` like Part 5) if the pilot looks worth it.

**What this adds, and what it doesn't replace.** `docs/method.md` retired an
encoding-model *ablation* (fit full vs. reward-removed, permutation-test the
drop) for two reasons: collinearity inversion and being supervised-only by
construction. This cross-check reuses the same design-matrix machinery but
**not** the ablation — no reduced refit, no permutation test. It fits one
full ridge encoding model per session (stimulus, position, movement,
reward/lick/cue lag-bases, corridor landmarks, recovered from
`git show 0aa0138`) predicting each neuron's raw activity trace, then asks
whether the size of a neuron's reward-lag-basis weights (`‖coef_reward‖`)
agrees with its cross-validated `|d'|` from Part 4. Two structurally
different measures — one from moment-to-moment temporal dynamics, one from
trial-averaged cue timing — agreeing is stronger evidence than either alone;
and because the GLM holds position, movement, lick, and cue timing fixed,
agreement also argues that Part 4's `|d'|` isn't just riding one of those
confounds.

**Reward/cue collinearity, again.** Reward and the sound cue are tightly
coupled by task structure (the cue predicts reward) — the same collinearity
that retired the ablation. Ridge shrinks and splits weight between
correlated regressors, so the reward-only coefficient norm can under-read a
real effect; a combined reward+cue norm is reported alongside it as the
robustness check.

**Read `ρ` (Spearman), not `p`.** With ~3,000 neurons in the sample, p-values
are near-zero regardless of effect size — `ρ` is the number that means
something here.

**Verification.** The design-matrix and fitting code is covered by an
offline synthetic self-check (`run_encoding_synthetic_check`, no download
needed) that injects graded reward drive into fake neurons and confirms the
fitted coefficient norm recovers the ranking. The pilot cell itself needs the
Figshare download to run, same as the rest of the notebook, and has not been
run against real data as part of this edit.


In [ ]:
# --- Encoding-model design matrix: recovered from git history (commit 0aa0138).
# balanced_sample, map_regions, reconstruct_selected, download_by_name, AREA_CODES,
# N_PER_REGION, SEED, manifest, stage_beh, paired_animals, animal_id, and
# reward_dprime_session are reused unchanged from Parts 1-5 above; only the
# design-matrix / GLM machinery below is new.
from dataclasses import dataclass
import math

from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, GroupKFold
from sklearn.preprocessing import StandardScaler
from scipy.stats import spearmanr

FRAME_FIELDS = (
    "ft", "ft_trInd", "ft_trInd_odd", "ft_trInd_even", "ft_WallID",
    "ft_Pos", "ft_RunSpeed", "ft_isMoving", "ft_move", "BefCueFr",
    "AftCueFr", "ft_GraySpc", "ft_CorrSpc",
)
MAX_FRAME_SLACK = 10  # behavior logging may run a few samples past the imaging clock


@dataclass
class DesignMatrix:
    """The regressors, their names, and which columns belong to which block."""
    raw: np.ndarray  # frames x features, unscaled
    names: list[str]
    blocks: dict[str, np.ndarray]  # block name -> column indices into `raw`
    frame_rate: float


def align_behavior_frames(beh, n_frames):
    """Return a copy whose frame-aligned arrays match the number of neural frames.

    Sessions differ by a few terminal samples, so trim up to MAX_FRAME_SLACK excess
    behavior samples. Behavior shorter than the neural data, or a larger mismatch,
    means the streams are misaligned - fail loudly instead of shifting regressors.
    """
    aligned = dict(beh)
    trimmed = []
    for field in FRAME_FIELDS:
        values = np.asarray(beh[field])
        excess = len(values) - n_frames
        if excess == 0:
            aligned[field] = values
        elif 0 < excess <= MAX_FRAME_SLACK:
            aligned[field] = values[:n_frames]
            trimmed.append(field)
        else:
            raise ValueError(
                f"{field} has {len(values)} frames but neural data has {n_frames}; "
                f"only a terminal excess of <= {MAX_FRAME_SLACK} can be trimmed."
            )
    if trimmed:
        print(f"Trimmed {len(trimmed)} behavior fields to {n_frames} neural frames.")
    return aligned


def infer_frame_rate(ft):
    """Frame rate from timestamp spacing. Zhong stores MATLAB datenums (days)."""
    ft = np.asarray(ft, dtype=float)
    diffs = np.diff(ft[np.isfinite(ft)])
    dt = float(np.median(diffs[diffs > 0]))
    dt_seconds = dt * 86400.0 if dt < 0.01 else dt
    if not 0.01 <= dt_seconds <= 10.0:
        raise ValueError(f"Implausible neural frame interval: {dt_seconds:.6g} seconds")
    return 1.0 / dt_seconds


def _basis_centers(low, high, n_basis):
    """Evenly spaced bump centers, each bump overlapping its neighbours."""
    if not np.isfinite(low) or not np.isfinite(high) or high <= low or n_basis <= 0:
        raise ValueError(f"Invalid raised-cosine range: {low} to {high}, {n_basis} bases")
    centers = np.linspace(low, high, n_basis)
    width = max((high - low) / max(n_basis - 1, 1) * 1.5, np.finfo(float).eps)
    return centers, width


def _cosine_bump(distance):
    """1 at a basis centre, tapering smoothly to 0 at |distance| = 1."""
    values = np.zeros(distance.shape, dtype=np.float32)
    inside = np.isfinite(distance) & (np.abs(distance) <= 1)
    values[inside] = (0.5 + 0.5 * np.cos(np.pi * distance[inside])).astype(np.float32)
    return values


def raised_cosine_basis(values, low, high, n_basis):
    """Soft-bin a continuous variable (position, speed) into overlapping bumps."""
    centers, width = _basis_centers(low, high, n_basis)
    values = np.asarray(values, dtype=float)
    return _cosine_bump((values[:, None] - centers[None, :]) / width)


def event_basis(n_frames, events, frame_rate, start_s, end_s, n_basis):
    """Spread each event over lag bumps, so a neuron can respond before or after it."""
    if n_frames <= 0 or frame_rate <= 0:
        raise ValueError("Invalid event-basis dimensions")
    centers, width = _basis_centers(start_s, end_s, n_basis)
    output = np.zeros((int(n_frames), int(n_basis)), dtype=np.float32)
    events = np.asarray(events, dtype=float).ravel()
    for event in events[np.isfinite(events)]:
        first = max(0, int(math.floor(event + start_s * frame_rate)))
        last = min(n_frames - 1, int(math.ceil(event + end_s * frame_rate)))
        if last < first:
            continue  # the event's whole window falls outside the recording
        frames = np.arange(first, last + 1)
        lags = (frames - event) / frame_rate
        output[frames] += _cosine_bump((lags[:, None] - centers[None, :]) / width)
    return output


def odd_even_masks(frame_trials):
    """Use first/third/fifth/... trials to train, matching Zhong's 'odd' mask."""
    frame_trials = np.asarray(frame_trials, dtype=float)
    valid = np.isfinite(frame_trials)
    trial_ids = np.unique(np.rint(frame_trials[valid]).astype(np.int64))
    if len(trial_ids) < 2:
        raise ValueError("At least two trials are required")
    train = valid & np.isin(np.rint(frame_trials), trial_ids[::2])
    test = valid & np.isin(np.rint(frame_trials), trial_ids[1::2])
    if not train.any() or not test.any():
        raise ValueError("Odd/even trial split is empty")
    return train, test


def _append_block(arrays, names, blocks, block_name, values, column_names):
    """Add one group of regressors and remember which columns it occupies."""
    values = np.asarray(values, dtype=np.float32)
    if values.ndim == 1:
        values = values[:, None]
    start = sum(array.shape[1] for array in arrays)
    arrays.append(values)
    names.extend(column_names)
    blocks[block_name] = np.arange(start, start + values.shape[1], dtype=int)


def build_design_matrix(beh, train_mask, reward_events=None):
    """Build every regressor: stimulus, position, movement, reward/lick/cue lag-bases,
    trial epoch, and corridor landmarks. Only the speed range is estimated from data,
    and only from training frames, so the held-out trials stay untouched.
    """
    required = [
        "ft", "ft_WallID", "ft_Pos", "ft_RunSpeed", "ft_isMoving",
        "ft_trInd", "LickFr", "RewardFr", "BefCueFr", "AftCueFr",
        "ft_GraySpc", "ft_CorrSpc", "StartFr", "GrayFr", "EndFr",
    ]
    missing = [field for field in required if field not in beh]
    if missing:
        raise KeyError(f"Missing behavior fields: {missing}")
    n_frames = len(np.asarray(beh["ft"]))
    train_mask = np.asarray(train_mask, dtype=bool)
    frame_rate = infer_frame_rate(beh["ft"])
    arrays, names, blocks = [], [], {}

    wall = np.asarray(beh["ft_WallID"]).astype(str)
    stimuli = sorted(value for value in np.unique(wall) if value.lower() != "nan")
    if len(stimuli) < 2:
        raise ValueError(f"Expected at least two stimuli, found {stimuli}")
    stimulus = np.column_stack([wall == value for value in stimuli]).astype(np.float32)
    _append_block(arrays, names, blocks, "stimulus", stimulus,
                  [f"stimulus:{value}" for value in stimuli])

    position = np.asarray(beh["ft_Pos"], dtype=float)
    position_basis = raised_cosine_basis(position, 0.0, 60.0, 12)
    _append_block(arrays, names, blocks, "position", position_basis,
                  [f"position_basis:{i}" for i in range(12)])
    interactions = (stimulus[:, :, None] * position_basis[:, None, :]).reshape(n_frames, -1)
    _append_block(
        arrays, names, blocks, "position_x_stimulus", interactions,
        [f"position_x_stimulus:{stim}:{i}" for stim in stimuli for i in range(12)],
    )

    speed = np.asarray(beh["ft_RunSpeed"], dtype=float)
    train_speed = speed[train_mask & np.isfinite(speed)]
    speed = np.where(np.isfinite(speed), speed, np.median(train_speed))
    low, high = np.percentile(train_speed, [1, 99])
    speed_basis = raised_cosine_basis(speed, float(low), float(max(high, low + 1.0)), 5)
    movement_values = np.column_stack([
        speed,
        speed_basis,
        np.asarray(beh["ft_isMoving"], dtype=np.float32),
        np.diff(speed, prepend=speed[0]),
    ])
    _append_block(
        arrays, names, blocks, "movement", movement_values,
        ["running_speed"] + [f"speed_basis:{i}" for i in range(5)]
        + ["is_moving", "acceleration"],
    )

    if reward_events is None:
        reward_events = np.asarray(beh["RewardFr"], dtype=float)
    reward = event_basis(n_frames, reward_events, frame_rate, -3.0, 3.0, 8)
    _append_block(arrays, names, blocks, "reward", reward,
                  [f"reward_lag_basis:{i}" for i in range(8)])

    lick = event_basis(n_frames, beh["LickFr"], frame_rate, -1.0, 2.0, 8)
    _append_block(arrays, names, blocks, "lick", lick,
                  [f"lick_lag_basis:{i}" for i in range(8)])

    cue_field = "SoundDelayFr" if "SoundDelayFr" in beh else "SoundFr"
    cue = event_basis(n_frames, beh[cue_field], frame_rate, 0.0, 3.0, 8)
    _append_block(arrays, names, blocks, "cue", cue,
                  [f"cue_lag_basis:{i}" for i in range(8)])

    epoch = np.column_stack([
        beh["BefCueFr"], beh["AftCueFr"], beh["ft_GraySpc"], beh["ft_CorrSpc"],
    ]).astype(np.float32)
    _append_block(arrays, names, blocks, "epoch", epoch,
                  ["before_cue", "after_cue", "gray_space", "corridor_space"])

    for source, label in [("StartFr", "start"), ("GrayFr", "gray"), ("EndFr", "end")]:
        landmark = event_basis(n_frames, beh[source], frame_rate, 0.0, 2.0, 6)
        _append_block(arrays, names, blocks, f"landmark_{label}", landmark,
                      [f"landmark_{label}_basis:{i}" for i in range(6)])

    raw = np.column_stack(arrays).astype(np.float32, copy=False)
    if not np.isfinite(raw).all():
        bad = np.flatnonzero(~np.isfinite(raw).all(axis=0))
        raise ValueError(f"Design matrix has non-finite columns: {bad[:10].tolist()}")
    return DesignMatrix(raw=raw, names=names, blocks=blocks, frame_rate=frame_rate)


def regression_metrics(y_true, y_pred):
    """Held-out error and variance explained, one value per neuron."""
    mse = mean_squared_error(y_true, y_pred, multioutput="raw_values")
    r2 = r2_score(y_true, y_pred, multioutput="raw_values")
    return mse, r2


def run_encoding_synthetic_check():
    """Inject graded reward drive into several fake neurons; the reward-coef norm
    must rank them in the same order as the injected gain, or the pipeline is broken.
    """
    rng = np.random.default_rng(11)
    n_trials, frames_per_trial = 50, 15
    n_frames = n_trials * frames_per_trial
    trials = np.repeat(np.arange(n_trials), frames_per_trial)
    train, _ = odd_even_masks(trials.astype(float))
    events = np.arange(n_trials) * frames_per_trial + rng.integers(3, frames_per_trial - 3, n_trials)

    reward = event_basis(n_frames, events, 5.0, -1.0, 1.0, 4)
    nuisance = rng.normal(size=(n_frames, 3))
    raw = np.column_stack([nuisance, reward]).astype(np.float32)
    reward_idx = np.arange(3, 7)
    scaler = StandardScaler().fit(raw[train])
    X = scaler.transform(raw)

    n_neurons = 8
    gains = rng.uniform(0.0, 4.0, n_neurons)  # ground-truth reward drive, graded
    Y = (gains[None, :] * reward[:, [2]] + 0.3 * nuisance[:, [0]]
        + rng.normal(0, 0.2, (n_frames, n_neurons))).astype(np.float32)

    model = Ridge(alpha=0.1).fit(X[train], Y[train])
    coef_norm = np.linalg.norm(model.coef_[:, reward_idx], axis=1)
    rho, _ = spearmanr(coef_norm, gains)
    if not (rho > 0.8):
        raise AssertionError(f"Encoding-model synthetic check failed: rho={rho:.3f}")
    return {"spearman_rho": round(float(rho), 3)}


In [ ]:
# --- Pilot run: one supervised, post-learning session ---
GLM_COHORT, GLM_STAGE = "supervised", "train2_after"
GLM_ALPHAS = np.array([0.1, 1.0, 10.0, 100.0, 1000.0])

print("Encoding-model synthetic check:", run_encoding_synthetic_check())

glm_animal = paired_animals[GLM_COHORT][0]
glm_beh_all = stage_beh[(GLM_COHORT, GLM_STAGE)]
glm_mouse = next(m for m in glm_beh_all if animal_id(m) == glm_animal)
glm_beh = glm_beh_all[glm_mouse]
print(f"Pilot session: {glm_mouse} ({GLM_COHORT}/{GLM_STAGE})")

if np.isfinite(np.asarray(glm_beh["RewardFr"], dtype=float)).sum() < 10:
    raise ValueError(f"{glm_mouse}: too few reward events for the encoding-model fit")

# Same neuron sample the d' pipeline used for this session: reward_dprime_session
# below makes an identical balanced_sample call (same regions, n_per_region, seed),
# so its neuron order matches glm_sel_regions - verified by the assert further down.
with warnings.catch_warnings():
    warnings.filterwarnings("ignore", message="Trying to unpickle estimator")
    glm_svd = np.load(download_by_name(f"{glm_mouse}_SVD_dec.npy", manifest),
                      allow_pickle=True).item()
glm_retin = np.load(download_by_name(f"{glm_mouse[:-1]}trans.npz", manifest), allow_pickle=False)
glm_sel, glm_sel_regions = balanced_sample(map_regions(glm_retin["iarea"]), N_PER_REGION, seed=SEED)
glm_Y = reconstruct_selected(glm_svd, glm_sel)
del glm_svd

glm_n_common = min(glm_Y.shape[0], len(np.asarray(glm_beh["ft"])))
glm_Y = glm_Y[:glm_n_common]
glm_beh_aligned = align_behavior_frames(glm_beh, glm_n_common)

glm_train_mask, glm_test_mask = odd_even_masks(glm_beh_aligned["ft_trInd"])
glm_design = build_design_matrix(glm_beh_aligned, glm_train_mask)
glm_scaler = StandardScaler().fit(glm_design.raw[glm_train_mask])
glm_X = glm_scaler.transform(glm_design.raw)
glm_reward_idx = glm_design.blocks["reward"]
glm_reward_cue_idx = np.concatenate([glm_design.blocks["reward"], glm_design.blocks["cue"]])

glm_trial_ids = np.nan_to_num(np.asarray(glm_beh_aligned["ft_trInd"], dtype=float), nan=-1).astype(np.int64)
glm_train_groups = glm_trial_ids[glm_train_mask]

print("Selecting ridge alpha (grouped folds inside training trials)...")
glm_search = GridSearchCV(Ridge(), {"alpha": GLM_ALPHAS}, cv=GroupKFold(n_splits=3),
                          scoring="neg_mean_squared_error", refit=False)
glm_search.fit(glm_X[glm_train_mask], glm_Y[glm_train_mask], groups=glm_train_groups)
glm_best_alpha = float(glm_search.best_params_["alpha"])
glm_alphas_tried = np.array(glm_search.cv_results_["param_alpha"], dtype=float)
glm_val_mse = -glm_search.cv_results_["mean_test_score"]
print(f"Selected alpha={glm_best_alpha:g}")

glm_model = Ridge(alpha=glm_best_alpha).fit(glm_X[glm_train_mask], glm_Y[glm_train_mask])
glm_mse_test, glm_r2_test = regression_metrics(glm_Y[glm_test_mask], glm_model.predict(glm_X[glm_test_mask]))

# Reward-encoding strength per neuron, two ways: reward lag-basis alone, and
# reward+cue combined. Reward and cue are the two regressors most tightly coupled
# by task structure (the cue predicts reward) - the exact collinearity that retired
# the encoding-model ablation (docs/method.md). Ridge shrinks and splits weight
# between correlated regressors, so the reward-only norm can under-read a real
# effect; the combined-block norm is the robustness check against that.
glm_reward_strength = np.linalg.norm(glm_model.coef_[:, glm_reward_idx], axis=1)
glm_reward_cue_strength = np.linalg.norm(glm_model.coef_[:, glm_reward_cue_idx], axis=1)

# Cross-validated |d'| for the SAME neurons (Part 4's index).
glm_dprime_res = reward_dprime_session(glm_beh, glm_mouse, manifest)
glm_dprime_strength = np.concatenate([glm_dprime_res["strength_by_region"][r] for r in AREA_CODES])
glm_regions_flat = np.concatenate([[r] * len(glm_dprime_res["strength_by_region"][r]) for r in AREA_CODES])
assert np.array_equal(glm_sel_regions, glm_regions_flat), \
    "Neuron sample order mismatch between the GLM and the d' pipeline"

glm_rho_reward, _ = spearmanr(glm_reward_strength, glm_dprime_strength)
glm_rho_combined, _ = spearmanr(glm_reward_cue_strength, glm_dprime_strength)
print(f"Held-out fit: median R²={np.nanmedian(glm_r2_test):.3f}, "
     f"median MSE={np.median(glm_mse_test):.4g}")
print(f"Spearman(reward-coef norm, cross-val |d'|)      = {glm_rho_reward:.3f}")
print(f"Spearman(reward+cue-coef norm, cross-val |d'|)  = {glm_rho_combined:.3f}")


In [ ]:
# --- Plots: alpha loss curve, held-out fit quality, and the cross-check scatter ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

ax = axes[0]
ax.semilogx(glm_alphas_tried, glm_val_mse, marker="o")
ax.axvline(glm_best_alpha, color="tab:red", ls="--", label=f"selected {glm_best_alpha:g}")
ax.set(xlabel="ridge alpha", ylabel="mean validation MSE\n(3-fold, training frames only)",
       title="Regularization curve\n(ridge has no iterative loss - this is its loss curve)")
ax.legend(fontsize=9)

ax = axes[1]
ax.hist(glm_r2_test[np.isfinite(glm_r2_test)], bins=30, color="0.4")
ax.axvline(0, color="tab:red", ls="--", lw=1)
ax.set(xlabel="held-out R² (odd/even trial split)", ylabel="neuron count",
       title=f"Fit quality, {glm_mouse}\nmedian R²={np.nanmedian(glm_r2_test):.3f}")

ax = axes[2]
for region, color in zip(AREA_CODES, sns.color_palette("deep", len(AREA_CODES))):
    m = glm_sel_regions == region
    ax.scatter(glm_reward_strength[m], glm_dprime_strength[m], s=14, alpha=0.6,
               color=color, label=region)
ax.set(xlabel="GLM reward-coef norm  ||coef_reward||",
       ylabel="cross-validated |d’| (late vs early cue)",
       title=("Do the two methods agree?\n"
              f"ρ(reward)={glm_rho_reward:.2f}, ρ(reward+cue)={glm_rho_combined:.2f}"))
ax.legend(fontsize=9, title="region")

fig.suptitle(f"Encoding-model cross-check — {glm_mouse} ({GLM_COHORT}/{GLM_STAGE})", y=1.04)
plt.tight_layout()
plt.show()
